[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

### My notes:

- Each token has its top-k given router logits, e.g.:

token 0 -> [expert 1, expert 3]
\
token 1 -> [expert 0, expert 2]
\
token 2 -> [expert 2, expert 3]
\
...
\
token 7 -> [expert 0, expert 1]

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        # pass  # router + experts
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model)
            )
            for _ in range(num_experts)
        ])
        # normal mlp:
        # self.mlp = nn.Sequential(
        #         nn.Linear(d_model, 4 * d_model),
        #         nn.ReLU(),
        #         nn.Linear(4 * d_model, d_model)
        #     ) 
        self.top_k = top_k


    def forward(self, x):
        # pass  # route tokens to top-k experts
        B, T, D = x.shape
        router_logits = self.router(x) # [B, T, num_experts]

        topk_logits, topk_indices = torch.topk(
            router_logits,
            self.top_k,
            dim=-1
        )
        print(topk_logits.shape, topk_indices.shape)  # [B, T, K], [B, T, K]
        topk_weights = torch.softmax(topk_logits, dim=-1)   # [B, T, K]
        topk_weights_flat = topk_weights.reshape(-1)
        
        x_flat = x.reshape(B * T, D)
        topk_indices_flat = topk_indices.reshape(-1)
        
        # print(topk_indices_flat)
        out = torch.zeros_like(x_flat)   # [B*T, D]
        
        
        token_ids = torch.arange(B * T, device=x.device).repeat_interleave(self.top_k)
        # print(token_ids[topk_indices_flat == 0])
        # token_ids[topk_indices_flat == i] : token indices of token that is routed to expert i
        for i, expert in enumerate(self.experts):
            mask = topk_indices_flat == i
            
            if not mask.any():
                continue
    
            expert_out = expert(x_flat[token_ids[mask]]) # token_ids[mask] : token indices of token that is routed to expert i

            
            weights = topk_weights_flat[mask].unsqueeze(-1)

            # expert 0 output: [5, D], expert 1 output: [3, D] ... expert_out.size[0]: number of tokens routed to expert i
            out[token_ids[mask]] += expert_out * weights  # weighted sum of expert outputs
            
        # print(out.shape)
        out = out.reshape(B, T, D)
        return out

In [6]:
# 🧪 Debug
torch.manual_seed(0)
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Output:', moe(x))
print('Params:', sum(p.numel() for p in moe.parameters()))

torch.Size([2, 8, 2]) torch.Size([2, 8, 2])
Output: torch.Size([2, 8, 32])
torch.Size([2, 8, 2]) torch.Size([2, 8, 2])
Output: tensor([[[ 0.0557, -0.1019, -0.1481, -0.0356, -0.3420,  0.2064,  0.0227,
          -0.0840,  0.0958,  0.0914, -0.0121, -0.0542,  0.1493, -0.0701,
          -0.2293,  0.1346,  0.2565, -0.1894,  0.2095,  0.0327, -0.1826,
           0.0741,  0.2349,  0.1000,  0.1472,  0.3201, -0.1744,  0.2456,
           0.0862, -0.0813,  0.0405, -0.0052],
         [ 0.1014,  0.0359, -0.2515, -0.3029, -0.3821,  0.0990, -0.0333,
          -0.1882,  0.0693, -0.1244,  0.0591,  0.2977,  0.1009,  0.2493,
           0.1610, -0.1713,  0.2468, -0.1627, -0.0162, -0.3432,  0.0653,
          -0.0505, -0.0169,  0.2942, -0.1935,  0.4072,  0.0244,  0.4502,
          -0.0168, -0.0013, -0.3512, -0.2192],
         [ 0.1478,  0.2678, -0.0634, -0.0297,  0.1013, -0.1344,  0.0105,
           0.1426, -0.0762,  0.2244,  0.1771, -0.0211, -0.0508, -0.0240,
           0.0216, -0.0081,  0.1347, -0.0135,  0.

In [98]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.4ms)
  ✅ [2/4] Has router and experts (0.5ms)
  ✅ [3/4] Router logits shape (0.7ms)
  ✅ [4/4] Gradient flow (1.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (3.9ms total)
  Progress saved. Run status() to see your dashboard.

